# 02 · Template-matching baseline

Simplest thing that could work: `cv2.matchTemplate` to slide a frame over the map. It only works when scale and rotation already agree, so it will not survive a real flight — but it gives you a number to beat and exercises the whole measure-the-error loop end to end.

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from navmod import matcher, camera, metrics, georef

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"


def show(img, title=None, figsize=(11, 7)):
    """Inline display for OpenCV arrays (handles BGR->RGB and grayscale)."""
    plt.figure(figsize=figsize)
    gray = img.ndim == 2
    plt.imshow(img if gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
               cmap="gray" if gray else None)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


print("ready ·", cv2.__version__)

## Generate a frame with known truth

`simulation.fake_drone_frame` at heading 0, scale 1. Because you chose (x, y), you know the answer.

## Match and read off the peak

`cv2.matchTemplate` + `cv2.minMaxLoc`. Mind the convention: the result is the *corner* of the best window, and your truth is probably the *centre*. Off-by-half-a-frame is the classic bug here.

Compare `TM_CCOEFF_NORMED` against `TM_SQDIFF_NORMED` and note which extreme you want from `minMaxLoc` for each.

## Error in metres

`metrics.error_m`. Also look at the peak value and how sharply it stands above the surrounding response — a flat response surface means a weak fix, and that ratio is your first confidence signal.

## Find the breaking point

Sweep heading away from 0, then scale away from 1, and watch error explode. Record roughly where it fails — that threshold is exactly what feature matching in notebook 03 is meant to fix.